In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/MyDrive/PKG - C-NMC 2019.zip" -d /content/cnmc_dataset

In [ ]:
import os
os.listdir('/content/cnmc_dataset')

In [ ]:
os.listdir('/content/cnmc_dataset/PKG - C-NMC 2019')

In [ ]:
os.listdir('/content/cnmc_dataset/PKG - C-NMC 2019/C-NMC_training_data')

In [ ]:
os.listdir('/content/cnmc_dataset/PKG - C-NMC 2019/C-NMC_training_data/fold_0')

In [ ]:
import os
from pathlib import Path
from PIL import Image
from collections import defaultdict, Counter

base = Path('/content/cnmc_dataset/PKG - C-NMC 2019/C-NMC_training_data')

folds = ['fold_0', 'fold_1', 'fold_2']
classes = ['all', 'hem']

total_images = 0
bad_files = []
extensions = Counter()
sizes = Counter()
summary = []

for fold in folds:
    for cls in classes:
        folder = base / fold / cls

        files = [f for f in folder.iterdir() if f.is_file() and not f.name.startswith('.')]
        count = len(files)
        total_images += count

        for file in files:
            extensions[file.suffix.lower()] += 1
            try:
                img = Image.open(file)
                sizes[img.size] += 1
                img.verify()
            except Exception as e:
                bad_files.append(str(file))

        summary.append((fold, cls, count))

print("="*60)
print("DATASET SUMMARY")
print("="*60)

for row in summary:
    print(f"Fold: {row[0]:8s}  Class: {row[1]:4s}  Images: {row[2]}")

print("\nTotal Images:", total_images)

print("\nFile Extensions:")
for k,v in extensions.items():
    print(k, ":", v)

print("\nTop 10 Image Sizes:")
for k,v in sizes.most_common(10):
    print(k, ":", v)

print("\nCorrupt Files Found:", len(bad_files))

if bad_files:
    print("\nSample Bad Files:")
    for f in bad_files[:10]:
        print(f)

In [ ]:
# ============================================================
# PHASE 02 : DATA PIPELINE + PREPROCESSING + MANUAL HOLDOUT
# Leukemia Project | C-NMC 2019 | ConvNeXt Large Ready
# ============================================================

# ----------------------------
# 1. INSTALL / IMPORTS
# ----------------------------
!pip -q install timm

import os
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# ----------------------------
# 2. REPRODUCIBILITY
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# 3. PATHS
# ----------------------------
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = Path('/content/cnmc_dataset/PKG - C-NMC 2019/C-NMC_training_data')

# !unzip -q "/content/drive/MyDrive/PKG - C-NMC 2019.zip" -d /content/cnmc_dataset

DRIVE_ROOT = Path('/content/drive/MyDrive')
MANUAL_DIR = DRIVE_ROOT / 'manual_testing'

# ----------------------------
# 4. SETTINGS
# ----------------------------
IMG_SIZE = 384
BATCH_SIZE = 16          # H100 can increase later (32/64)
NUM_WORKERS = 2
PIN_MEMORY = True

# Holdout images (untouched forever)
MANUAL_PER_CLASS = 100   # 100 ALL + 100 HEM

CLASS_MAP = {
    'hem': 0,   # healthy
    'all': 1    # leukemia
}

# ----------------------------
# 5. CREATE MASTER DATAFRAME
# ----------------------------
records = []

for fold in ['fold_0', 'fold_1', 'fold_2']:
    for cls in ['all', 'hem']:
        folder = BASE_PATH / fold / cls
        for file in folder.glob('*.bmp'):
            records.append({
                'filepath': str(file),
                'filename': file.name,
                'fold': fold,
                'class_name': cls,
                'label': CLASS_MAP[cls]
            })

df = pd.DataFrame(records)

print("="*60)
print("MASTER DATASET")
print("="*60)
print(df.head())
print("\nTotal Images:", len(df))
print(df['class_name'].value_counts())

# ----------------------------
# 6. CREATE UNTOUCHED MANUAL HOLDOUT
# ----------------------------
# Delete old folder if exists
if MANUAL_DIR.exists():
    shutil.rmtree(MANUAL_DIR)

(MANUAL_DIR / 'all').mkdir(parents=True, exist_ok=True)
(MANUAL_DIR / 'hem').mkdir(parents=True, exist_ok=True)

manual_parts = []

for cls in ['all', 'hem']:
    temp = df[df['class_name'] == cls].copy()

    # stratified across folds manually
    selected = (
        temp.groupby('fold', group_keys=False)
        .apply(lambda x: x.sample(
            n=max(1, round(MANUAL_PER_CLASS / 3)),
            random_state=SEED
        ))
    )

    # fix exact count
    if len(selected) > MANUAL_PER_CLASS:
        selected = selected.sample(MANUAL_PER_CLASS, random_state=SEED)

    while len(selected) < MANUAL_PER_CLASS:
        remain = temp[~temp['filepath'].isin(selected['filepath'])]
        extra = remain.sample(1, random_state=SEED)
        selected = pd.concat([selected, extra], ignore_index=True)

    manual_parts.append(selected)

manual_df = pd.concat(manual_parts, ignore_index=True)

# copy files to drive folder
for _, row in manual_df.iterrows():
    src = Path(row['filepath'])
    dst = MANUAL_DIR / row['class_name'] / src.name
    shutil.copy2(src, dst)

# save manifest
manual_df.to_csv(MANUAL_DIR / 'manual_testing_manifest.csv', index=False)

# remove from main dataframe
df = df[~df['filepath'].isin(manual_df['filepath'])].reset_index(drop=True)

print("\n" + "="*60)
print("MANUAL TESTING SET CREATED")
print("="*60)
print(manual_df['class_name'].value_counts())
print("Saved to:", MANUAL_DIR)

# ----------------------------
# 7. VERIFY ZERO OVERLAP
# ----------------------------
overlap = set(df['filepath']).intersection(set(manual_df['filepath']))
print("\nOverlap with training pool:", len(overlap))

# ----------------------------
# 8. TRAIN / VALIDATION SPLIT USING FOLDS
# CURRENT DEFAULT:
# Train = fold_0 + fold_1
# Val   = fold_2
# ----------------------------
train_df = df[df['fold'].isin(['fold_0', 'fold_1'])].reset_index(drop=True)
val_df   = df[df['fold'].isin(['fold_2'])].reset_index(drop=True)

print("\n" + "="*60)
print("TRAIN / VAL SPLIT")
print("="*60)
print("Train:", len(train_df))
print("Val:", len(val_df))

print("\nTrain Class Counts")
print(train_df['class_name'].value_counts())

print("\nVal Class Counts")
print(val_df['class_name'].value_counts())

# ----------------------------
# 9. CLASS WEIGHTS
# ----------------------------
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1]),
    y=train_df['label'].values
)

class_weights = torch.tensor(weights, dtype=torch.float32)

print("\nClass Weights:", class_weights)

# ----------------------------
# 10. IMAGE TRANSFORMS
# ----------------------------
train_transform = transforms.Compose([
    transforms.Resize((420,420)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

# ----------------------------
# 11. CUSTOM DATASET
# ----------------------------
class CNMCDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row['filepath']).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row['label'], dtype=torch.long)

        return img, label

# ----------------------------
# 12. DATASETS
# ----------------------------
train_dataset = CNMCDataset(train_df, train_transform)
val_dataset   = CNMCDataset(val_df, val_transform)

# ----------------------------
# 13. WEIGHTED SAMPLER
# ----------------------------
sample_weights = train_df['label'].map({
    0: class_weights[0].item(),
    1: class_weights[1].item()
}).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ----------------------------
# 14. DATALOADERS
# ----------------------------
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# ----------------------------
# 15. SANITY CHECK
# ----------------------------
x, y = next(iter(train_loader))

print("\n" + "="*60)
print("PIPELINE READY")
print("="*60)
print("Train batch image shape :", x.shape)   # [B,3,384,384]
print("Train batch labels      :", y.shape)
print("Unique labels in batch  :", torch.unique(y))

# ----------------------------
# 16. SAVE CSV FILES
# ----------------------------
PROJECT_META = DRIVE_ROOT / "Leukemia_Project_Metadata"
PROJECT_META.mkdir(exist_ok=True)

train_df.to_csv(PROJECT_META / "train_df.csv", index=False)
val_df.to_csv(PROJECT_META / "val_df.csv", index=False)
df.to_csv(PROJECT_META / "remaining_pool.csv", index=False)

print("\nSaved CSV metadata to:", PROJECT_META)

# ============================================================
# END OF PHASE 02
# Ready for Phase 03 (ConvNeXt Large Training)
# ============================================================

In [ ]:
# ============================================================
# PHASE 03 : TRAINING PIPELINE (H100 OPTIMIZED)
# Leukemia Project | ConvNeXt Large | Save Best Model to Drive
# Uses train_loader / val_loader / class_weights from Phase 02
# ============================================================

# ----------------------------
# 1. INSTALL / IMPORTS
# ----------------------------
!pip -q install timm torchmetrics

import os
import copy
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ============================================================
# 2. CONFIG
# ============================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_NAME = 'convnext_large'
NUM_CLASSES = 2

EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 5  # early stopping

SAVE_DIR = Path('/content/drive/MyDrive/Leukemia_Project_Models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = SAVE_DIR / 'best_convnext_large_leukemia.pth'
LAST_MODEL_PATH = SAVE_DIR / 'last_convnext_large_leukemia.pth'
HISTORY_CSV = SAVE_DIR / 'training_history.csv'

print("Device:", DEVICE)

# ============================================================
# 3. MODEL
# ============================================================
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# ============================================================
# 4. LOSS / OPTIMIZER / SCHEDULER
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = GradScaler()

# ============================================================
# 5. HELPERS
# ============================================================
def train_one_epoch(model, loader):
    model.train()

    running_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.softmax(outputs, dim=1)[:,1]
        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds, all_probs)

    return epoch_loss, metrics


def validate_one_epoch(model, loader):
    model.eval()

    running_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:,1]
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds, all_probs)

    return epoch_loss, metrics


def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = 0.0

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc
    }

# ============================================================
# 6. TRAIN LOOP
# ============================================================
history = []

best_f1 = 0
best_epoch = 0
patience_counter = 0

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    print("="*70)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("="*70)

    train_loss, train_metrics = train_one_epoch(model, train_loader)
    val_loss, val_metrics = validate_one_epoch(model, val_loader)

    scheduler.step()

    lr_now = optimizer.param_groups[0]['lr']

    row = {
        'epoch': epoch,
        'lr': lr_now,

        'train_loss': train_loss,
        'train_acc': train_metrics['accuracy'],
        'train_f1': train_metrics['f1'],
        'train_auc': train_metrics['auc'],

        'val_loss': val_loss,
        'val_acc': val_metrics['accuracy'],
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1'],
        'val_auc': val_metrics['auc']
    }

    history.append(row)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_metrics['accuracy']:.4f}")
    print(f"Val F1     : {val_metrics['f1']:.4f}")
    print(f"Val AUC    : {val_metrics['auc']:.4f}")
    print(f"LR         : {lr_now:.7f}")

    # ----------------------------
    # SAVE BEST MODEL
    # ----------------------------
    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        best_epoch = epoch
        patience_counter = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'model_name': MODEL_NAME
        }, BEST_MODEL_PATH)

        print(">>> BEST MODEL SAVED TO DRIVE")

    else:
        patience_counter += 1

    # save last model every epoch
    torch.save(model.state_dict(), LAST_MODEL_PATH)

    # save history
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {PATIENCE} non-improving epochs.")
        break

# ============================================================
# 7. TRAINING COMPLETE
# ============================================================
total_time = (time.time() - start_time) / 60

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Best Epoch : {best_epoch}")
print(f"Best Val F1: {best_f1:.4f}")
print(f"Time Taken : {total_time:.2f} min")
print(f"Best Model : {BEST_MODEL_PATH}")

# ============================================================
# 8. PLOTS
# ============================================================
hist_df = pd.DataFrame(history)

plt.figure(figsize=(10,5))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss')
plt.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss')
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10,5))
plt.plot(hist_df['epoch'], hist_df['val_f1'], label='Val F1')
plt.plot(hist_df['epoch'], hist_df['val_auc'], label='Val AUC')
plt.title("Validation Metrics")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.show()

# ============================================================
# 9. LOAD BEST MODEL READY FOR NEXT PHASE
# ============================================================
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("\nBest model loaded and ready for Phase 04.")
# ============================================================

In [ ]:
# ============================================================
# NEW PHASE 04 : FINAL EVALUATION + MANUAL TESTING
# Leukemia Project | ConvNeXt Large
# Loads best saved model from Drive
# Generates metrics + confusion matrix + ROC + manual testing
# ============================================================

# ----------------------------
# 1. INSTALL / IMPORTS
# ----------------------------
!pip -q install timm seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# ============================================================
# 2. CONFIG
# ============================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_PATH = Path('/content/drive/MyDrive/Leukemia_Project_Models/best_convnext_large_leukemia.pth')

VAL_CSV = Path('/content/drive/MyDrive/Leukemia_Project_Metadata/val_df.csv')

MANUAL_DIR = Path('/content/drive/MyDrive/manual_testing')

OUTPUT_DIR = Path('/content/drive/MyDrive/Leukemia_Project_Results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
IMG_SIZE = 384

CLASS_NAMES = ['hem', 'all']   # 0,1

print("Device:", DEVICE)

# ============================================================
# 3. TRANSFORM
# ============================================================
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std =[0.229,0.224,0.225]
    )
])

# ============================================================
# 4. DATASET CLASS
# ============================================================
class LeukemiaDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row['filepath']).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = int(row['label'])

        return img, label


# ============================================================
# 5. LOAD VALIDATION DATA
# ============================================================
val_df = pd.read_csv(VAL_CSV)

val_dataset = LeukemiaDataset(val_df, val_transform)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Validation Samples:", len(val_df))

# ============================================================
# 6. LOAD MODEL
# ============================================================
model = timm.create_model(
    'convnext_large',
    pretrained=False,
    num_classes=2
)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()

print("Best model loaded.")

# ============================================================
# 7. VALIDATION PREDICTION
# ============================================================
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)[:,1]
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# ============================================================
# 8. METRICS
# ============================================================
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {auc:.4f}")

# Save metrics
metrics_df = pd.DataFrame([{
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1_score': f1,
    'roc_auc': auc
}])

metrics_df.to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)

# ============================================================
# 9. CLASSIFICATION REPORT
# ============================================================
report = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4
)

print("\nCLASSIFICATION REPORT\n")
print(report)

with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)

# ============================================================
# 10. CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300)
plt.show()

# ============================================================
# 11. ROC CURVE
# ============================================================
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=300)
plt.show()

# ============================================================
# 12. MANUAL TESTING SET PREDICTION
# ============================================================
manual_records = []

for cls_name in ['hem', 'all']:

    folder = MANUAL_DIR / cls_name

    for file in folder.glob('*.bmp'):

        img = Image.open(file).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(img_tensor)
            prob = torch.softmax(output, dim=1)[0][1].item()
            pred = torch.argmax(output, dim=1).item()

        manual_records.append({
            'filename': file.name,
            'true_class': cls_name,
            'true_label': 0 if cls_name == 'hem' else 1,
            'predicted_class': CLASS_NAMES[pred],
            'predicted_label': pred,
            'leukemia_probability': prob
        })

manual_df = pd.DataFrame(manual_records)

manual_df.to_csv(OUTPUT_DIR / 'manual_testing_predictions.csv', index=False)

manual_acc = accuracy_score(
    manual_df['true_label'],
    manual_df['predicted_label']
)

manual_f1 = f1_score(
    manual_df['true_label'],
    manual_df['predicted_label']
)

print("\n" + "="*60)
print("MANUAL TESTING RESULTS")
print("="*60)
print("Samples:", len(manual_df))
print(f"Accuracy: {manual_acc:.4f}")
print(f"F1 Score: {manual_f1:.4f}")

# ============================================================
# 13. TOP WRONG PREDICTIONS
# ============================================================
wrong_df = manual_df[
    manual_df['true_label'] != manual_df['predicted_label']
]

wrong_df.to_csv(OUTPUT_DIR / 'manual_wrong_predictions.csv', index=False)

print("Wrong Predictions:", len(wrong_df))

# ============================================================
# 14. FINAL SAVE SUMMARY
# ============================================================
summary = f"""
Leukemia Project Final Evaluation

Validation Accuracy : {acc:.4f}
Validation F1 Score : {f1:.4f}
Validation ROC AUC  : {auc:.4f}

Manual Testing Accuracy : {manual_acc:.4f}
Manual Testing F1 Score : {manual_f1:.4f}
"""

with open(OUTPUT_DIR / 'final_summary.txt', 'w') as f:
    f.write(summary)

print("\nSaved all outputs to:")
print(OUTPUT_DIR)

# ============================================================
# END OF PHASE 04
# ============================================================

In [ ]:
# ============================================================
# LEUKEMIA DETECTION APP (COLAB NOTEBOOK VERSION)
# Upload image -> preprocess -> load saved model -> predict
# ============================================================

!pip -q install timm

import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from google.colab import files

# ============================================================
# 1. CONFIG
# ============================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_PATH = "/content/drive/MyDrive/Leukemia_Project_Models/best_convnext_large_leukemia.pth"

IMG_SIZE = 384

CLASS_NAMES = {
    0: "Healthy (HEM)",
    1: "Leukemia (ALL)"
}

print("Device:", DEVICE)

# ============================================================
# 2. LOAD MODEL
# ============================================================
model = timm.create_model(
    "convnext_large",
    pretrained=False,
    num_classes=2
)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()

print("Model loaded successfully.")

# ============================================================
# 3. PREPROCESSING
# ============================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

# ============================================================
# 4. UPLOAD IMAGE
# ============================================================
print("\nUpload blood smear image (.bmp/.jpg/.png)")
uploaded = files.upload()

# ============================================================
# 5. PREDICT
# ============================================================
for filename in uploaded.keys():

    # Load image
    image = Image.open(filename).convert("RGB")

    # Show image
    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.title("Uploaded Image")
    plt.axis("off")
    plt.show()

    # preprocess
    img_tensor = transform(image).unsqueeze(0).to(DEVICE)

    # inference
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)[0]

    pred_class = torch.argmax(probs).item()

    healthy_prob = probs[0].item() * 100
    leukemia_prob = probs[1].item() * 100

    print("="*60)
    print("PREDICTION RESULT")
    print("="*60)
    print("Predicted Class :", CLASS_NAMES[pred_class])
    print(f"Healthy Confidence  : {healthy_prob:.2f}%")
    print(f"Leukemia Confidence: {leukemia_prob:.2f}%")

    if pred_class == 1:
        print("\n⚠️ Model Prediction: Leukemia-like Cell Detected")
    else:
        print("\n✅ Model Prediction: Healthy Cell Detected")